In [151]:
import pandas as pd
from rapidfuzz import fuzz, process
import json

In [152]:
record_formats = pd.read_csv("data/csv/Moving_record_formats_v2.csv")

In [153]:
print_types = []
print_columns = []
for i in record_formats.T:
    print_types.append(record_formats.loc[i, "print type"])
    columns = record_formats.loc[i][[
            "sarakkeet",
            "Unnamed: 7",
            "Unnamed: 8",
            "Unnamed: 9",
            "Unnamed: 10",
            "Unnamed: 11",
            "Unnamed: 12",
            "Unnamed: 13",
            "Unnamed: 14",
            "Unnamed: 15",
            "Unnamed: 16",
            "Unnamed: 17",
            "Unnamed: 18",
            "Unnamed: 19",
            "Unnamed: 20",
            "Unnamed: 21",
            "Unnamed: 22",
            "Unnamed: 23",
            "Unnamed: 24",
            "Unnamed: 25",
            "Unnamed: 26",
            "Unnamed: 27",
            "Unnamed: 28",
            "Unnamed: 29",
            "Unnamed: 30",
            "Unnamed: 31",
            "Unnamed: 32",
            "Unnamed: 33",
            "Unnamed: 34",
            "Unnamed: 35",
        ]].dropna().to_list()
    print_columns.append(columns)

record_formats = record_formats[
    ["print type", "pages", "in/out", "sarakkeiden lkm", "sarakkeet"]
]
record_formats["sarakkeet"] = print_columns

#record_formats["sarakkeiden lkm"] = record_formats["sarakkeet"].apply(
    #lambda x: len(x)
#)

In [154]:
record_formats["sarakkeiden lkm"] = record_formats[
    ["sarakkeiden lkm", "sarakkeet"]
].apply(
    lambda x: (
        int(from_string)
        if (from_string := x["sarakkeiden lkm"].split(" ")[-1]).isnumeric()
        else len(x["sarakkeet"])
    ),
    axis=1,
)

In [155]:
record_formats

,print type,pages,in/out,sarakkeiden lkm,sarakkeet
0,print 1,one table,in,14,"[kuukausi, päivä, nimi ja sääty, miespuoli, na..."
1,print 17,one table,out,14,"[kuukausi, päivä, muuttokirjan numero, nimi ja..."
2,print 2,two tables,in,11,"[juokseva numero, muuttokirja sisäänannettiin(..."
3,print 2,NaN,in,11,"[juokseva numero, muuttokirja sisäänannettiin(..."
4,print 3,two tables,in,9,"[kuukausi, päivä, aleslin numero, tulleitten s..."
...,...,...,...,...,...
89,print 53,one table,?,47,[]
90,print 54,two tables,out,8,"[år och månad, dag, betygets nummer, de utflyt..."
91,print 54,NaN,out,8,"[år och månad, dag, betygets nummer, de utflyt..."
92,print 55,two tables,in,9,"[(muuttokirjan)n:o, (muuttokirjan)n:o, (muutto..."


In [156]:
useful_cols = []
col_search_words = {
    "name": ["nimi", "nimet", "namn"],
    "birthday": ["syntymä", "syntymäpäivä", "födels", "födelsdag"]
}

for cols in print_columns:
    useful_cols_idxs = {
        "name": -1,
        "birthday": -1
    }


    for c, search_words in col_search_words.items():
        results = [process.extractOne(word, cols, scorer=fuzz.partial_ratio) for word in search_words]
        if results[0] == None:
            continue
        result = max(results, key=lambda x: x[1])
        if result[1] > 80:
            useful_cols_idxs[c] = cols.index(result[0])

    useful_cols.append(useful_cols_idxs)

In [157]:
useful_cols_dict = {k: v for k, v in zip(print_types, useful_cols)}

with open("outputs/json/useful_cols.json", "w") as fp:
    json.dump(useful_cols_dict, fp)